# MORSE from Python

Run a small CPU experiment, inspect its Pareto candidates and reopen
the saved results. The bundled scorer measures CpG content and GC
distance; these demonstration metrics do not predict expression.

Install MORSE with the notebook extra following the repository README,
then select that environment as the notebook kernel. No additional
models or services are needed for this example.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shlex
import sys

import yaml
import morse

# A new folder for each independent run.
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
experiment = Path.cwd() / "results" / f"python_api_demo_{stamp}"
experiment.mkdir(parents=True, exist_ok=False)
print(experiment)


## Input and configuration

The input contains a 5′UTR, CDS body, 3′UTR and poly(A), without
start or stop codons. Schema 5 adds these from separate lists.
Coordinates cover the input consecutively using 0-based,
end-exclusive ranges. The UTR libraries provide two choices each.


In [ ]:
utr5 = "GGGAAACCC"
cds_body = "".join(["ACU", "UGU", "GGU", "ACU", "AAG",
                    "CUU", "GGU", "AGU", "GGU", "AGU"])
utr3 = "GGAUCUCC"
rna_seq = utr5 + cds_body + utr3 + "A" * 12

(experiment / "sequence.fasta").write_text(f">demo_rna\n{rna_seq}\n")
(experiment / "utr5.fasta").write_text(
    ">leader_a\nGGGAAACCC\n>leader_b\nGGAACUCC\n")
(experiment / "utr3.fasta").write_text(
    ">trailer_a\nGGAUCUCC\n>trailer_b\nCCUAAGGA\n")

cds_start = len(utr5)
cds_end = cds_start + len(cds_body)
utr3_end = cds_end + len(utr3)
params = {
    "schema_version": 5,
    "name": "python_api_demo",
    "output_dir": "run",
    "construct": {
        "fasta": "sequence.fasta",
        "start_codon": ["AUG"],
        "stop_codon": ["UAA"],
        "coordinates": {
            "five_prime_utr": [0, cds_start],
            "cds": [cds_start, cds_end],
            "three_prime_utr": [cds_end, utr3_end],
            "poly_a": [utr3_end, len(rna_seq)],
        },
        "changes": {
            "cds": "synonymous",
            "five_prime_utr": {"library": "utr5.fasta"},
            "three_prime_utr": {"library": "utr3.fasta"},
        },
    },
    "optimizer": {
        "algorithm": "nsga2", "population_size": 12,
        "generations": 3, "seed": 43,
    },
    "scorers": [{
        "name": "sequence_metrics",
        "command": f"{shlex.quote(sys.executable)} -m morse.resources.simple_scorer",
        "scope": "cds",
        "metrics": {
            "cpg_per_kb": {"role": "optimize", "direction": "min"},
            "gc_distance": {"role": "optimize", "direction": "min"},
            "gc_fraction": {"role": "log"},
        },
    }],
}
config_path = experiment / "experiment.yaml"
config_path.write_text(yaml.safe_dump(params, sort_keys=False))

# Resolve paths relative to the YAML; the result is an ordinary dict.
params = morse.load_config(config_path)
print(f"Input: {len(rna_seq)} nt; configuration: {config_path}")


## Check the input and initial scores

Passing `rna_seq` uses this string instead of the configured FASTA.
The configuration supplies input coordinates, start/stop choices
and scorer settings. Scorer `scope: cds` receives the assembled CDS,
including the start and stop.


In [ ]:
check = morse.validate(rna_seq, params)
check


In [ ]:
scores = morse.score(rna_seq, params)
scores["outputs"]


## Optimize

`verbose=2` updates progress and plots inside the cell. Use `1` for
progress only or `0` for a quiet run. Results remain in the experiment
folder after the notebook session ends.


In [ ]:
result = morse.optimize(
    rna_seq, params, verbose=2,
    plots=["metrics", "hypervolume", "utrs"],
)
print("Results:", result.output_dir)


## Inspect and select

The Pareto set contains trade-offs between the objectives. Select one
candidate by exclusive hypervolume contribution, or by a named metric.


In [ ]:
print("Status:", result.summary["status"])
print("Pareto candidates:", len(result.pareto))
result.pareto[:3]


In [ ]:
candidate = result.select("hvc")
print(candidate.id)
print(candidate.sequence)
candidate.metrics


In [ ]:
# Lists of dictionaries can also be passed to pandas.DataFrame().
result.history[-1]


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

figure = result.plot("utrs")
display(figure)
plt.close(figure)


In [ ]:
result.report()


## Reopen saved results

Loading results does not rerun scorers. To continue an interrupted run,
use `morse.resume(result.output_dir, verbose=2)` in the same output folder.


In [ ]:
reopened = morse.load_run(result.output_dir)
print(reopened.summary["status"])
reopened.pareto[:2]
